# Imports

In [ ]:
import harbor.analysis.cross_docking as cd
from importlib import reload
import pandas as pd
reload(cd)

# load data

In [ ]:
data = cd.DockingDataModel.deserialize("/Users/alexpayne/Scientific_Projects/mers-drug-discovery/sars2-retrospective-analysis/full_cross_dock_v2_combined_results/ALL_1_poses.json")

In [ ]:
df = data.dataframe

In [ ]:
df

In [ ]:
data.get_groupby_columns()

In [ ]:
df[df.duplicated(data.get_groupby_columns())]

In [ ]:
df[data.get_groupby_columns()].isna().sum()

In [ ]:
missing_chem_sim_data = df[df[data.get_groupby_columns()].isna().any(axis=1)]

In [ ]:
missing_chem_sim_data.nunique()

In [ ]:
query_refs = {(query, ref) for query, ref in zip(missing_chem_sim_data.Query_Ligand, missing_chem_sim_data.Reference_Ligand)}

In [ ]:
query_refs

In [ ]:
chem_sim_df = pd.read_csv("/Users/alexpayne/Scientific_Projects/mers-drug-discovery/sars2-retrospective-analysis/full_cross_dock_v2_chemical_similarity_data/combined_chemical_similarity_data.csv")

In [ ]:
# check to see if the missing data is in the chemical similarity data
missing_chem_sim_data[missing_chem_sim_data["Query_Ligand"].isin(chem_sim_df["Query_Ligand"])]

In [ ]:
chem_sim_df[chem_sim_df["Query_Ligand"].isin(missing_chem_sim_data["Query_Ligand"])]

# Figure out what evaluator is doing

In [ ]:
test_data = data.model_copy()

In [ ]:
ev = cd.Evaluator.from_json_file(
    "/Users/alexpayne/Scientific_Projects/mers-drug-discovery/sars2-retrospective-analysis/test_failed_analysis/evaluator_reference_split_comparison_18.json")
ev.n_bootstraps = 10
ev.run(test_data)

In [ ]:
test_data == data

In [ ]:
data.dataframe.nunique()

In [ ]:
test_data.dataframe.nunique()

In [ ]:
test_data.dataframe.nunique()

In [ ]:
d1 = ev.run_pose_selector([test_data])

In [ ]:
d1[0].dataframe.nunique()

## why does the pose selector do this

In [ ]:
newdata = test_data.model_copy()
key_columns = newdata.get_groupby_columns(except_cols=["Pose_ID"])
sf = cd.ColumnSortFilter(
    sort_column="Pose_ID",
    key_columns=key_columns,
    ascending=True,
    number_to_return=1,
)
newdata.apply_filters([sf])

In [ ]:
filtered = test_data.dataframe.groupby(key_columns).head(1)

In [ ]:
missing_q = set(test_data.dataframe.Query_Ligand.unique()) - set(filtered.Query_Ligand.unique())

In [ ]:
missing_r = set(test_data.dataframe.Reference_Ligand.unique()) - set(filtered.Reference_Ligand.unique())

In [ ]:
missing_q

In [ ]:
missing_r

### make sure in chem sim data

In [ ]:
chem_sim_df[chem_sim_df["Query_Ligand"].isin(missing_q)]

In [ ]:
chem_sim_df[chem_sim_df["Reference_Ligand"].isin(missing_q)]

# this is the problem

## which rows are missing in d1?

In [ ]:
d2 = ev.run_dataset_split(d1)

In [ ]:
d2[0].dataframe.nunique()

In [ ]:
d3 = ev.run_scorer(d2)

In [ ]:
d3[0].dataframe.nunique()